In [1]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px 


In [2]:
df = pd.read_parquet('../data/processed/clean_data.parquet')
print(df.shape)
print(df.head())

(590540, 360)
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...  id_20    id_28     id_29  \
0  361.0  150.0    discover  142.0  ...  472.0  Unknown   Unknown   
1  404.0  150.0  mastercard  102.0  ...  472.0  Unknown   Unknown   
2  490.0  150.0        visa  166.0  ...  472.0  Unknown   Unknown   
3  567.0  150.0  mastercard  117.0  ...  472.0  Unknown   Unknown   
4  514.0  150.0  mastercard  102.0  ...  144.0      New  NotFound   

                 id_31    id_35    id_36    id_37    id_38  DeviceType  \
0              Unknown  Unknown  U

In [3]:
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day'] = (df['TransactionDT'] // 86400) % 7
df['is_peak_hour'] = df['hour'].apply(lambda x: 1 if 6 <= x <= 9 else 0)

df['amt_log'] = np.log1p(df['TransactionAmt'])

anonymous_emails = ['protonmail.com', 'mail.com', 'outlook.es', 'aim.com']
df['is_anonymous_email'] = df['P_emaildomain'].apply(
    lambda x: 1 if x in anonymous_emails else 0)

df['is_mobile'] = df['DeviceType'].apply(
    lambda x: 1 if x == 'mobile' else 0)

df['is_discover'] = df['card4'].apply(
    lambda x: 1 if x == 'discover' else 0)

df['is_high_risk_product'] = df['ProductCD'].apply(
    lambda x: 1 if x == 'C' else 0)

print(df[['hour', 'day', 'is_peak_hour', 'amt_log',
          'is_anonymous_email', 'is_mobile', 
          'is_discover', 'is_high_risk_product']])

        hour  day  is_peak_hour   amt_log  is_anonymous_email  is_mobile  \
0          0    1             0  4.241327                   0          0   
1          0    1             0  3.401197                   0          0   
2          0    1             0  4.094345                   0          0   
3          0    1             0  3.931826                   0          0   
4          0    1             0  3.931826                   0          1   
...      ...  ...           ...       ...                 ...        ...   
590535    23    0             0  3.912023                   0          0   
590536    23    0             0  3.701302                   0          0   
590537    23    0             0  3.464172                   0          0   
590538    23    0             0  4.770685                   0          0   
590539    23    0             0  5.638177                   0          0   

        is_discover  is_high_risk_product  
0                 1                     0  

In [4]:
card_agg = df.groupby("card1")["TransactionAmt"].agg(["mean", "std", "count"])

df['card1_amt_mean'] = df['card1'].map(card_agg['mean'])
df['card1_amt_std'] = df['card1'].map(card_agg['std'])
df['card1_count'] = df['card1'].map(card_agg['count'])

email_agg = df.groupby("P_emaildomain")["TransactionAmt"].agg(["mean", "std", "count"])

df['email_amt_mean'] = df['P_emaildomain'].map(email_agg['mean'])
df['email_amt_std'] = df['P_emaildomain'].map(email_agg['std'])
df['email_count'] = df['P_emaildomain'].map(email_agg['count'])

df['card1_amt_std'] = df['card1_amt_std'].fillna(0)
df['email_amt_std'] = df['email_amt_std'].fillna(0)

print(df[['card1_amt_mean', 'card1_amt_std', 'card1_count',
          'email_amt_mean', 'email_amt_std', 'email_count']].head())

   card1_amt_mean  card1_amt_std  card1_count  email_amt_mean  email_amt_std  \
0      351.931163     371.141254           43      119.092949     193.098963   
1      234.292753     460.356975          683      128.630195     227.044011   
2       97.015542     100.128858         1108      112.939682     214.141837   
3      123.416340     192.717425         4209      144.201108     274.237537   
4       96.972222      56.629451           18      128.630195     227.044011   

   email_count  
0        94456  
1       228355  
2         5096  
3       100934  
4       228355  


In [5]:
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns found:", categorical_columns)
print("Total:", len(categorical_columns))

label_encoder = LabelEncoder()

for column in categorical_columns:
    df[column + '_encoded'] = label_encoder.fit_transform(df[column].astype(str))

print(df.select_dtypes(include=['object']).columns.tolist())  

Categorical columns found: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_31', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
Total: 26
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_31', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [6]:
v_columns = [column for column in df.columns if column.startswith("V")]
print("Number of V columns:", len(v_columns))

df[v_columns] = df[v_columns].astype(np.float32)

scaler = StandardScaler()
v_scaled = scaler.fit_transform(df[v_columns])

pca_check = PCA()
pca_check.fit(v_scaled)

cumulative_explained_variance = pca_check.explained_variance_ratio_.cumsum()

fig = px.line(x=range(1, len(cumulative_explained_variance)+1),
              y=cumulative_explained_variance * 100,
              title='Cumulative Variance Explained by PCA Components',
              labels={'x': 'Number of Components', 'y': 'Variance Explained (%)'})
fig.add_hline(y=95, line_dash='dash', line_color='red', annotation_text='95%')
fig.show()

n_95 = next(i for i, v in enumerate(cumulative_explained_variance) if v >= 0.95) + 1
print(f"Components for 95% variance: {n_95}")

Number of V columns: 292


Components for 95% variance: 85


In [7]:
pca = PCA(n_components=n_95)
v_pca = pca.fit_transform(v_scaled)

print(f"Variance explained: {pca.explained_variance_ratio_.cumsum()[-1]*100:.2f}%")

pca_df = pd.DataFrame(v_pca, columns=[f'V_pca_{i}' for i in range(n_95)])
df = pd.concat([df.reset_index(drop=True), pca_df], axis=1)

df = df.drop(columns=v_columns)
print(df.shape)


Variance explained: 95.08%
(590540, 193)


In [8]:
cols_to_drop = [
    'TransactionID',     
    'TransactionDT',     
    'TransactionAmt',   
] + categorical_columns

df = df.drop(columns=cols_to_drop)

In [9]:
df.to_parquet('../data/processed/features.parquet', index=False)
print("Final shape:", df.shape)
print("Columns:", df.columns.tolist())

Final shape: (590540, 164)
Columns: ['isFraud', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'id_01', 'id_02', 'id_05', 'id_06', 'id_11', 'id_13', 'id_17', 'id_19', 'id_20', 'hour', 'day', 'is_peak_hour', 'amt_log', 'is_anonymous_email', 'is_mobile', 'is_discover', 'is_high_risk_product', 'card1_amt_mean', 'card1_amt_std', 'card1_count', 'email_amt_mean', 'email_amt_std', 'email_count', 'ProductCD_encoded', 'card4_encoded', 'card6_encoded', 'P_emaildomain_encoded', 'R_emaildomain_encoded', 'M1_encoded', 'M2_encoded', 'M3_encoded', 'M4_encoded', 'M5_encoded', 'M6_encoded', 'M7_encoded', 'M8_encoded', 'M9_encoded', 'id_12_encoded', 'id_15_encoded', 'id_16_encoded', 'id_28_encoded', 'id_29_encoded', 'id_31_encoded', 'id_35_encoded', 'id_36_encoded', 'id_37_encoded', 'id_38_encoded', 'DeviceType_encoded', 'DeviceInfo_encoded', 'V_pca_